# 뮤직비디오 감정(Valence·Arousal) 추출

Essentia의 공식 DEAM 모델(MusiCNN embedding)로 곡의 감정을 나타내는 valence(긍정도)·arousal(각성도) 값을 추출하는 코드입니다. 유튜브 URL에서 오디오를 다운로드해 MusiCNN 임베딩을 뽑고, DEAM 분류 헤드로 1~9 범위의 valence/arousal 값을 예측합니다.

**구성**
1. 환경 설정 — 필요한 패키지 설치
2. 메인 추출 — 곡 목록을 대상으로 오디오 다운로드, 임베딩 추출, valence/arousal 예측. mp3 로컬 저장과 중간/최종 CSV 저장 기능 포함

**참고사항**
- 이 코드는 Google Colab 환경(`google.colab.files`)을 기준으로 작성되었습니다.
- 사용 모델: Essentia 공식 사전학습 모델 (MusiCNN embedding + DEAM 분류 헤드)
  - Alonso-Jiménez et al. (2020), *TensorFlow Audio Models in Essentia*, ICASSP.
  - Aljanaki et al. (2017), *DEAM dataset*, PLoS ONE.
- 원본 곡 목록과 추출 결과 파일은 저작권이 있는 곡 정보를 포함하고 있어 이 저장소에는 포함하지 않았습니다.


## 1. 환경 설정

Essentia(TensorFlow 버전)와 오디오 다운로드/처리에 필요한 패키지(`yt-dlp`, `pandas`, `numpy`, `ffmpeg`)를 설치합니다.


In [ ]:
!pip install essentia-tensorflow yt-dlp pandas numpy -q
!apt-get install -y ffmpeg > /dev/null 2>&1

## 2. 메인 추출

`EssentiaArousalValenceMusiCNN` 클래스로 각 곡의 유튜브 URL에서 오디오를 다운로드하고, MusiCNN 임베딩 → DEAM 분류 헤드를 거쳐 valence/arousal 값을 예측합니다. mp3는 로컬에 저장하고, 10곡마다 중간 CSV를 저장하며, 완료 후 최종 CSV를 다운로드합니다.


In [ ]:
"""
✅ Essentia 공식 DEAM 모델 (MusiCNN embedding)
- 출력: valence, arousal (1-9 범위)
- mp3 로컬 저장 + 중간 CSV + 최종 CSV 다운로드 기능 포함
"""

# ============================================================
# Colab 설치 (첫 번째 셀)
# ============================================================
"""
!pip install essentia-tensorflow yt-dlp pandas numpy -q
!apt-get install -y ffmpeg > /dev/null 2>&1
print("✅ 설치 완료!")
"""

# ============================================================
# 메인 코드
# ============================================================

import essentia.standard as es
import numpy as np
import pandas as pd
import yt_dlp
import os
import tempfile
import time
from urllib.parse import urlparse, parse_qs
from google.colab import files
import shutil
import urllib.request
from pathlib import Path

class EssentiaArousalValenceMusiCNN:
    """Essentia 공식 DEAM 모델 (MusiCNN embedding)"""

    EMBEDDING_URL = 'https://essentia.upf.edu/models/feature-extractors/musicnn/msd-musicnn-1.pb'
    DEAM_URL = 'https://essentia.upf.edu/models/classification-heads/deam/deam-msd-musicnn-2.pb'

    def __init__(self):
        self.temp_dir = tempfile.mkdtemp()
        self.models_dir = os.path.join(self.temp_dir, 'models')
        os.makedirs(self.models_dir, exist_ok=True)

        print(f"\n{'='*70}")
        print(f"🎵 Essentia DEAM Arousal/Valence (MusiCNN)")
        print(f"{'='*70}")
        print(f"Embedding: MusiCNN (음악 전용)")
        print(f"출력: Valence, Arousal (1-9 범위)")
        print(f"{'='*70}\n")

    def download_model(self, url, filename):
        filepath = os.path.join(self.models_dir, filename)
        if os.path.exists(filepath):
            print(f"  ✓ 이미 존재: {filename}")
            return filepath
        print(f"  📥 다운로드 중: {filename} ...")
        try:
            urllib.request.urlretrieve(url, filepath)
            print(f"  ✓ 다운로드 완료!")
            return filepath
        except Exception as e:
            print(f"  ⚠️ 다운로드 실패: {e}")
            return None

    def setup_models(self):
        print("\n🔧 모델 다운로드 중...\n")
        self.embedding_path = self.download_model(self.EMBEDDING_URL, 'msd-musicnn-1.pb')
        self.deam_path = self.download_model(self.DEAM_URL, 'deam-msd-musicnn-2.pb')
        if not self.embedding_path or not self.deam_path:
            raise RuntimeError("❌ 모델 다운로드 실패")
        print("\n✅ 모델 준비 완료!\n")

    def extract_video_id(self, url):
        try:
            parsed_url = urlparse(url)
            if 'youtube.com' in parsed_url.netloc:
                return parse_qs(parsed_url.query).get('v', [None])[0]
            elif 'youtu.be' in parsed_url.netloc:
                return parsed_url.path[1:]
        except:
            return None
        return None

    def download_audio(self, url, output_path):
        ydl_opts = {
            'format': 'bestaudio/best',
            'postprocessors': [{
                'key': 'FFmpegExtractAudio',
                'preferredcodec': 'mp3',
                'preferredquality': '192',
            }],
            'outtmpl': output_path,
            'quiet': True,
            'no_warnings': True,
        }
        try:
            with yt_dlp.YoutubeDL(ydl_opts) as ydl:
                ydl.download([url])
            mp3_path = output_path + '.mp3'
            return mp3_path if os.path.exists(mp3_path) else None
        except Exception as e:
            print(f"    ⚠️ 다운로드 실패: {e}")
            return None

    def predict_arousal_valence(self, audio_path):
        try:
            print(f"    🎵 오디오 로딩 (16kHz)...")
            audio = es.MonoLoader(filename=audio_path, sampleRate=16000, resampleQuality=4)()
            print(f"    🧠 MusiCNN embedding 추출 중...")
            embedding_model = es.TensorflowPredictMusiCNN(
                graphFilename=self.embedding_path,
                output="model/dense/BiasAdd"
            )
            embeddings = embedding_model(audio)
            print(f"    🎯 Arousal/Valence 예측 중...")
            deam_model = es.TensorflowPredict2D(
                graphFilename=self.deam_path,
                output="model/Identity"
            )
            predictions = deam_model(embeddings)
            valence_arousal = np.mean(predictions, axis=0)
            valence_raw = float(valence_arousal[0])
            arousal_raw = float(valence_arousal[1])
            valence_normalized = (valence_raw - 1) / 8
            arousal_normalized = (arousal_raw - 1) / 8
            return {
                'valence_raw': valence_raw,
                'arousal_raw': arousal_raw,
                'valence_normalized': valence_normalized,
                'arousal_normalized': arousal_normalized,
                'model': 'DEAM (MusiCNN)',
                'range': '[1, 9]'
            }
        except Exception as e:
            print(f"    ⚠️ 예측 오류: {e}")
            import traceback
            traceback.print_exc()
            return None

    def process_url(self, url, save_audio_dir=None):
        video_id = self.extract_video_id(url)
        if not video_id:
            return None

        temp_audio_base = os.path.join(self.temp_dir, f"{video_id}")
        result = None

        try:
            print(f"    📥 다운로드 중...")
            audio_path = self.download_audio(url, temp_audio_base)
            if not audio_path:
                return None

            # mp3 로컬 저장
            if save_audio_dir:
                os.makedirs(save_audio_dir, exist_ok=True)
                dst_path = os.path.join(save_audio_dir, os.path.basename(audio_path))
                shutil.copy(audio_path, dst_path)
                print(f"    💾 로컬 저장됨: {dst_path}")

            result = self.predict_arousal_valence(audio_path)

            if os.path.exists(audio_path):
                os.remove(audio_path)

            return result
        except Exception as e:
            print(f"    ⚠️ 오류: {e}")
            return None

    def process_dataframe(self, df, url_col='url', save_audio_dir=None, save_intermediate_csv=None):
        total = len(df)
        results = []

        print(f"\n{'='*70}")
        print(f"🎵 {total}개 곡 처리 시작 (DEAM + MusiCNN)")
        print(f"{'='*70}\n")

        start_time = time.time()

        for idx, row in df.iterrows():
            url = row[url_col]
            print(f"[{idx + 1}/{total}] {url}")
            result = self.process_url(url, save_audio_dir=save_audio_dir)

            if result:
                print(f"    ✅ Arousal: {result['arousal_raw']:.2f} → {result['arousal_normalized']:.3f}")
                print(f"       Valence: {result['valence_raw']:.2f} → {result['valence_normalized']:.3f}")

            results.append(result if result else {})

            # 중간 CSV 저장
            if save_intermediate_csv and (idx + 1) % 10 == 0:
                temp_df = pd.DataFrame(results)
                temp_df.to_csv(save_intermediate_csv, index=False, encoding='utf-8-sig')
                print(f"    📝 중간 CSV 저장: {save_intermediate_csv}")

        results_df = pd.DataFrame(results)
        final_df = pd.concat([df.reset_index(drop=True), results_df], axis=1)

        success = sum(1 for r in results if r)
        total_time = time.time() - start_time
        print(f"\n{'='*70}")
        print(f"✅ 완료: {success}/{total} ({total_time/60:.1f}분, {total_time/total:.1f}초/곡)")
        print(f"{'='*70}")

        return final_df

    def cleanup(self):
        if os.path.exists(self.temp_dir):
            shutil.rmtree(self.temp_dir)


# ============================================================
# 메인 실행
# ============================================================

def main():
    print("\n" + "="*70)
    print("🎵 Essentia DEAM Arousal/Valence (MusiCNN)")
    print("="*70)
    print("\n📂 CSV 파일을 업로드하세요:")

    uploaded = files.upload()
    if not uploaded:
        print("❌ 파일이 업로드되지 않았습니다.")
        return

    input_filename = list(uploaded.keys())[0]
    df = pd.read_csv(input_filename, encoding='utf-8-sig')
    print(f"\n✅ {len(df)}개 레코드 읽기 완료")

    url_col = 'url' if 'url' in df.columns else input("URL 컬럼 이름: ").strip()

    predictor = EssentiaArousalValenceMusiCNN()
    predictor.setup_models()

    try:
        # mp3 저장 폴더
        audio_save_dir = './downloaded_mp3'
        # 중간 CSV
        intermediate_csv = 'deam_intermediate.csv'

        result_df = predictor.process_dataframe(
            df,
            url_col=url_col,
            save_audio_dir=audio_save_dir,
            save_intermediate_csv=intermediate_csv
        )

        # 최종 CSV 저장
        final_csv = 'deam_musicnn_arousal_valence_FINAL.csv'
        result_df.to_csv(final_csv, index=False, encoding='utf-8-sig')
        print(f"✅ 최종 CSV 로컬 저장됨: {final_csv}")

        # Colab 다운로드
        files.download(final_csv)
        print(f"📥 다운로드 시작: {final_csv}")

    finally:
        predictor.cleanup()

if __name__ == "__main__":
    main()
